# NYC Traffic Volume Counts Bronze Ingestion

##  Purpose
This notebook ingests **NYC Traffic Volume Counts** data from the NYC Open Data API into a **Bronze Delta table**.  
It ensures reproducibility, schema consistency, and auditability by applying validation rules, provenance tagging, and deduplication.

---

## Phase 1: Setup
- Defines Lakehouse paths for Bronze, Silver, and Gold layers.
- Creates directories using Fabric’s file system API to ensure consistent storage structure.

---

## Phase 2: Data Fetching
**Function: `fetch_traffic_by_year(year, batch_size=50000)`**
- Queries NYC Open Data endpoint: `7ym2-wayt.json`.
- Fetches traffic counts for a given year in batches.
- Orders by month/day/hour/minute for reproducibility.
- Handles request failures gracefully.
- Returns all records as a list of JSON objects.
**Function: `json_to_csv(records)`**
- Converts JSON list into CSV string.
- Escapes commas and quotes.
- Produces a header row followed by record rows.
**Output File:**
- Path: `bronze_traffic/traffic_counts_<YYYY-MM-DD>.csv`

---

## Phase 3:  Bronze Validation
- Reads raw CSV into Spark DataFrame.
- Casts schema:
  - Integer: `RequestID`, `SegmentID`, `Yr`, `M`, `D`, `HH`, `MM`, `Vol`
  - String: `Boro`, `Direction`, `street`, `fromSt`, `toSt`, `WktGeom`
- Flags missing values:
  - `flag_missing_request`
  - `flag_missing_segment`
  - `flag_missing_boro`
- Combines flags → `is_valid`.
- Adds provenance metadata:
  - `source` = `"nyc-open-data-traffic"`
  - `ingest_run_id`
  - `ingested_at` (timestamp)
  - `ingest_date` (ISO date)

**Outputs:**
- **Good rows** → Valid records.
- **Quarantine rows** → Invalid records flagged for review.

---

## Phase 3a: Deduplication & Delta Write
- Checks existing `bronze_traffic_nyc` table for already ingested dates.
- Filters out duplicates by `ingest_date`.
- Deduplicates new batch on keys:
  - `RequestID`, `SegmentID`, `Yr`, `M`, `D`, `HH`, `MM`, `Direction`
- Appends good rows to **Bronze Delta table**:
  - Partitioned by `ingest_date` and `Boro`.
- Runs **safety net deduplication** across full table using Spark Window functions.
- Rewrites clean table if duplicates are found.
- Invalid rows written to `bronze_quarantine_traffic_nyc`.
- Partitioned by `ingest_date`.
- Ensures traceability of rejected records.

---


## Phase 4: QA Queries
The pipeline includes SQL queries to validate ingestion quality and provide audit trails:
1. **Total Count & Integrity**: total rows,  unique `RequestID` and `SegmentID`, ingestion runs.  
2. **Records by Borough**: Summarizes record counts, unique segments, and total traffic volume grouped by `Boro`. 
3. **Records by Direction**: Shows distribution of records across traffic directions (e.g., N, S, E, W).  
4. **Time Coverage**: Validates coverage by year and month, counting unique days and hours present.  
5. **Minute Distribution**: Ensures minutes (`MM`) are only in expected intervals: **0, 15, 30, 45**
6. **Null Checks**: Counts nulls in critical fields (`RequestID`, `SegmentID`, `Boro`, `Direction`, `Vol`).  
7. **Volume Statistics**:Provides min, avg, and max values of traffic volume (`Vol`) to check for anomalies.


### Phase 1: Setup

In [13]:
import datetime, uuid, requests, json, time

from functools import reduce
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, IntegerType, StringType
from pyspark.sql import functions as F
from datetime import date
from notebookutils import mssparkutils
from pyspark.sql import types as T
from datetime import date


from pyspark.sql import SparkSession
# Initialize Spark
spark = SparkSession.builder.getOrCreate()


StatementMeta(, 17eaf14e-5bc5-48f1-a6bb-30488a9025bd, 15, Finished, Available, Finished, False)

In [14]:
LAKEHOUSE = "test_lakehouse_1"   # replace with your actual Lakehouse name
BASE_PATH = "Files/project_data"  # Relative path - Fabric requirement

PATHS = {
    "bronze_weather": f"{BASE_PATH}/bronze/weather",
    "bronze_311":     f"{BASE_PATH}/bronze/311",
    "bronze_traffic": f"{BASE_PATH}/bronze/traffic",
    "silver":         f"{BASE_PATH}/silver",
    "gold":           f"{BASE_PATH}/gold",
}

# Create directories using Fabric's file system API with relative paths
for name, path in PATHS.items():
    try:
        mssparkutils.fs.mkdirs(path)
        #print(f" {name}: {path}")
    except Exception as e:
        print(f"{name} creation issue: {e}")

print(f"\nAll folders ready under {BASE_PATH}")

StatementMeta(, 17eaf14e-5bc5-48f1-a6bb-30488a9025bd, 16, Finished, Available, Finished, False)


All folders ready under Files/project_data


### Phase 2: Data Fetching

In [16]:
def fetch_traffic_by_year(year, batch_size=50000):
    """
    Fetch NYC Traffic Volume Counts data for a given year from the NYC Open Data API.

    Args:
        year : int
            The year of traffic data to fetch (e.g., 2026).
        batch_size : int, optional
            Number of records to request per API call (default = 50,000).
    Returns:
        list of dict
            A list of JSON records (Python dictionaries) containing traffic volume data
            for the specified year.
    Raises:
        Exception: If all retry attempts fail.
    """
    
    BASE_URL = "https://data.cityofnewyork.us/resource/7ym2-wayt.json"
    all_records = []
    offset = 0

    print(f"\n Fetching NYC Traffic Volume Counts for {year}...")

    while True:
        params = {
            "$limit": batch_size,
            "$offset": offset,
            "$where": f"Yr = {year}",
            "$order": "M ASC, D ASC, HH ASC, MM ASC"
        }
        try:
            response = requests.get(BASE_URL, params=params, timeout=120)
            response.raise_for_status()
            batch = response.json()
        except requests.exceptions.RequestException as e:
            print(f"   Request failed: {e}")
            break

        if not batch:
            break

        all_records.extend(batch)
        offset += batch_size
        print(f"   Year {year}: {len(all_records):,} records so far...")

        if len(batch) < batch_size:
            break

    print(f" Total records fetched for {year}: {len(all_records):,}")
    return all_records


def json_to_csv(records):
    """Convert list of JSON records to CSV string"""
    if not records:
        return ""
    
    headers = list(records[0].keys())
    csv_lines = [",".join(headers)]
    for record in records:
        row = [str(record.get(h, "")) for h in headers]
        row = [f'"{v}"' if "," in v or '"' in v else v for v in row]
        csv_lines.append(",".join(row))
    
    return "\n".join(csv_lines)


# Fetch 2026 only
today = datetime.date.today()
raw_traffic = fetch_traffic_by_year(2026)

# Convert to CSV and save
csv_content = json_to_csv(raw_traffic)
csv_path = f"{PATHS['bronze_traffic']}/traffic_counts_{today.isoformat()}.csv"
mssparkutils.fs.put(csv_path, csv_content, overwrite=True)

file_size_mb = len(csv_content.encode('utf-8')) / 1024 / 1024
print(f"\n Saved: {csv_path}")
print(f" File size: {file_size_mb:.1f} MB")
print(f" Record count: {len(raw_traffic):,}")

StatementMeta(, 17eaf14e-5bc5-48f1-a6bb-30488a9025bd, 18, Finished, Available, Finished, False)


 Fetching NYC Traffic Volume Counts for 2026...
   Year 2026: 9,120 records so far...
 Total records fetched for 2026: 9,120

 Saved: Files/project_data/bronze/traffic/traffic_counts_2026-04-23.csv
 File size: 1.2 MB
 Record count: 9,120


### Phase 3: Bronze Validation

In [17]:
def validate_traffic_bronze(csv_path, ingest_run_id):
    """
    Validate and enrich NYC traffic CSV data for Bronze ingestion.

    Reads a raw CSV into Spark, casts schema types, applies structural
    validation flags (missing RequestID, SegmentID, Boro), and adds
    provenance metadata. Splits the dataset into valid ("good") rows
    and invalid ("quarantine") rows.

    Args
        csv_path : str
            Path to the raw traffic CSV file.
        ingest_run_id : str
            Unique identifier for the ingestion run.

    Returns
        (DataFrame, DataFrame)
        Tuple of Spark DataFrames: (good, quarantine).
    """

    print(f"\n[BRONZE] Reading raw CSV: {csv_path}")
    df = spark.read.option("header", "true").option("inferSchema", "false").csv(csv_path)

    raw_count = df.count()
    print(f"  Raw rows: {raw_count:,}")

    # Cast schema
    df = (df.withColumn("RequestID", F.col("RequestID").cast(IntegerType()))
            .withColumn("SegmentID", F.col("SegmentID").cast(IntegerType()))
            .withColumn("Yr", F.col("Yr").cast(IntegerType()))
            .withColumn("M", F.col("M").cast(IntegerType()))
            .withColumn("D", F.col("D").cast(IntegerType()))
            .withColumn("HH", F.col("HH").cast(IntegerType()))
            .withColumn("MM", F.col("MM").cast(IntegerType()))
            .withColumn("Vol", F.col("Vol").cast(IntegerType()))
            .withColumn("Boro", F.col("Boro").cast(StringType()))
            .withColumn("Direction", F.col("Direction").cast(StringType()))
            .withColumn("street", F.col("street").cast(StringType()))
            .withColumn("fromSt", F.col("fromSt").cast(StringType()))
            .withColumn("toSt", F.col("toSt").cast(StringType()))
            .withColumn("WktGeom", F.col("WktGeom").cast(StringType()))
        )

    # Structural validation flags
    df = df.withColumn("flag_missing_request", F.col("RequestID").isNull())
    df = df.withColumn("flag_missing_segment", F.col("SegmentID").isNull())
    df = df.withColumn("flag_missing_boro", F.col("Boro").isNull())

    # Combine flags
    flag_names = [c for c in df.columns if c.startswith("flag_")]
    combined_flags = reduce(lambda acc, c: acc | F.col(c), flag_names, F.lit(False))
    df = df.withColumn("is_valid", ~combined_flags)

    # Provenance metadata
    ingest_date = date.today().isoformat()
    df = (df.withColumn("source", F.lit("nyc-open-data-traffic"))
            .withColumn("ingest_run_id", F.lit(ingest_run_id))
            .withColumn("ingested_at", F.current_timestamp())
            .withColumn("ingest_date", F.lit(ingest_date)))

    good = df.filter(F.col("is_valid") == True).drop("is_valid", *flag_names)
    quarantine = df.filter(F.col("is_valid") == False)

    print(f"  Good rows: {good.count():,}")
    print(f"  Quarantine rows: {quarantine.count():,}")

    return good, quarantine


StatementMeta(, 17eaf14e-5bc5-48f1-a6bb-30488a9025bd, 19, Finished, Available, Finished, False)

### Phase 3b: Deduplicaiton & Delta Write

In [18]:
ingest_run_id = "run_" + str(uuid.uuid4())
today_str = date.today().isoformat()
csv_path = f"{PATHS['bronze_traffic']}/traffic_counts_{today_str}.csv"

print(f"Processing Traffic Counts 2026...")

good, quarantine = validate_traffic_bronze(csv_path, ingest_run_id)

# Check what's already in the table 
try:
    df_existing = spark.table("bronze_traffic_nyc")
    existing_dates = set(
        row["ingest_date"] for row in
        df_existing.select("ingest_date").distinct().collect()
    )
    existing_count = df_existing.count()
    print(f"Existing bronze_traffic_nyc: {existing_count:,} rows")
    print(f"Already-ingested dates: {sorted(existing_dates)}")
except Exception:
    existing_dates = set()
    print("No existing table — first run.")

# Filter out rows for dates already ingested 
if existing_dates:
    good = good.filter(~F.col("ingest_date").isin(existing_dates))
    quarantine = quarantine.filter(~F.col("ingest_date").isin(existing_dates))

good_count = good.count()
quarantine_count = quarantine.count()
print(f"New good rows to write: {good_count:,}")
print(f"New quarantine rows to write: {quarantine_count:,}")

if good_count == 0:
    print("Already up to date — nothing to append.")
else:
    # Dedup within the new batch 
    good = good.dropDuplicates(["RequestID", "SegmentID", "Yr", "M", "D", "HH", "MM", "Direction"])

    print(f"Appending {good.count():,} good rows to bronze_traffic_nyc...")
    good.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .partitionBy("ingest_date", "Boro") \
        .saveAsTable("bronze_traffic_nyc")

    # Safety net: dedup the full table 
    print("Running dedup safety net on full table...")
    df_full = spark.table("bronze_traffic_nyc")
    before_dedup = df_full.count()

    w = Window.partitionBy("RequestID", "SegmentID", "Yr", "M", "D", "HH", "MM", "Direction") \
              .orderBy(F.desc("ingested_at"))
    df_deduped = df_full.withColumn("_rn", F.row_number().over(w)) \
                        .filter(F.col("_rn") == 1) \
                        .drop("_rn")
    after_dedup = df_deduped.count()

    if before_dedup != after_dedup:
        dupes = before_dedup - after_dedup
        print(f"Found {dupes:,} duplicates — rewriting clean table...")
        df_deduped.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .partitionBy("ingest_date", "Boro") \
            .saveAsTable("bronze_traffic_nyc")
        print(f"Table rewritten: {after_dedup:,} rows (removed {dupes:,} duplicates)")
    else:
        print(f"No duplicates found — table is clean ({after_dedup:,} rows)")

if quarantine_count > 0:
    print(f"Writing {quarantine_count:,} quarantine rows...")
    quarantine.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .partitionBy("ingest_date") \
        .saveAsTable("bronze_quarantine_traffic_nyc")
    print("Quarantine table updated")

print(f"\nTraffic ingestion complete! Run ID: {ingest_run_id}")


StatementMeta(, 17eaf14e-5bc5-48f1-a6bb-30488a9025bd, 20, Finished, Available, Finished, False)

Processing Traffic Counts 2026...

[BRONZE] Reading raw CSV: Files/project_data/bronze/traffic/traffic_counts_2026-04-23.csv
  Raw rows: 9,120
  Good rows: 9,120
  Quarantine rows: 0
Existing bronze_traffic_nyc: 9,120 rows
Already-ingested dates: ['2026-04-21']
New good rows to write: 9,120
New quarantine rows to write: 0
Appending 9,120 good rows to bronze_traffic_nyc...
Running dedup safety net on full table...
Found 9,120 duplicates — rewriting clean table...
Table rewritten: 9,120 rows (removed 9,120 duplicates)

Traffic ingestion complete! Run ID: run_30e413a6-4ddf-4e82-ac5b-cc16215182e4


### Phase 4: QA queries

In [19]:
# QA QUERIES FOR BRONZE LAYER - TRAFFIC

# 1. Total count and record integrity
spark.sql("""
    SELECT 
        COUNT(*) as total_records,
        COUNT(DISTINCT RequestID) as unique_requests,
        COUNT(DISTINCT SegmentID) as unique_segments,
        COUNT(DISTINCT ingest_run_id) as ingestion_runs
    FROM bronze_traffic_nyc
""").show()

# 2. Records by borough
print("\nRecords by borough:")
spark.sql("""
    SELECT 
        Boro,
        COUNT(*) as record_count,
        COUNT(DISTINCT SegmentID) as unique_segments,
        SUM(Vol) as total_volume
    FROM bronze_traffic_nyc
    GROUP BY Boro
    ORDER BY record_count DESC
""").show()

# 3. Records by direction
print("\nRecords by direction:")
spark.sql("""
    SELECT 
        Direction,
        COUNT(*) as record_count
    FROM bronze_traffic_nyc
    GROUP BY Direction
    ORDER BY record_count DESC
""").show()

# 4. Time coverage (year, month, day, hour)
print("\nTime coverage:")
spark.sql("""
    SELECT 
        Yr,
        M,
        COUNT(*) as records,
        COUNT(DISTINCT D) as unique_days,
        COUNT(DISTINCT HH) as unique_hours
    FROM bronze_traffic_nyc
    GROUP BY Yr, M
    ORDER BY Yr, M
""").show()

# 5. Minute distribution (should be 0, 15, 30, 45 only)
print("\nMinute distribution (should only see 0, 15, 30, 45):")
spark.sql("""
    SELECT 
        MM,
        COUNT(*) as record_count
    FROM bronze_traffic_nyc
    GROUP BY MM
    ORDER BY MM
""").show()

# 6. Null checks in critical fields
print("\nNull checks (should all be 0):")
spark.sql("""
    SELECT 
        SUM(CASE WHEN RequestID IS NULL THEN 1 ELSE 0 END) as null_request_id,
        SUM(CASE WHEN SegmentID IS NULL THEN 1 ELSE 0 END) as null_segment_id,
        SUM(CASE WHEN Boro IS NULL THEN 1 ELSE 0 END) as null_boro,
        SUM(CASE WHEN Direction IS NULL THEN 1 ELSE 0 END) as null_direction,
        SUM(CASE WHEN Vol IS NULL THEN 1 ELSE 0 END) as null_volume
    FROM bronze_traffic_nyc
""").show()

# 7. Volume statistics
print("\nVolume statistics:")
spark.sql("""
    SELECT 
        COUNT(*) as record_count,
        MIN(Vol) as min_volume,
        AVG(Vol) as avg_volume,
        MAX(Vol) as max_volume
    FROM bronze_traffic_nyc
""").show()


StatementMeta(, 17eaf14e-5bc5-48f1-a6bb-30488a9025bd, 21, Finished, Available, Finished, False)

+-------------+---------------+---------------+--------------+
|total_records|unique_requests|unique_segments|ingestion_runs|
+-------------+---------------+---------------+--------------+
|         9120|              6|             13|             1|
+-------------+---------------+---------------+--------------+


Records by borough:
+---------+------------+---------------+------------+
|     Boro|record_count|unique_segments|total_volume|
+---------+------------+---------------+------------+
|    Bronx|        5376|              4|      185087|
| Brooklyn|        2688|              4|      121040|
|Manhattan|         672|              1|       15527|
|   Queens|         384|              4|       24590|
+---------+------------+---------------+------------+


Records by direction:
+---------+------------+
|Direction|record_count|
+---------+------------+
|       NB|        2784|
|       SB|        2112|
|       EB|        2112|
|       WB|        2112|
+---------+------------+


Time 

<h2> Diagnostic </h2>

In [20]:
# ── DIAGNOSTIC 1: Raw Bronze profile ─────────────────────────────────────────

df_b = spark.table("bronze_traffic_nyc")

print(f"Total Bronze rows: {df_b.count():,}")
print(f"Unique RequestIDs: {df_b.select('RequestID').distinct().count():,}")
print(f"Unique SegmentIDs: {df_b.select('SegmentID').distinct().count():,}")
print()

print("── Boro values (raw, exactly as stored in Bronze) ──")
df_b.groupBy("Boro").count().orderBy(F.desc("count")).show(20, truncate=False)

print("── Unique segments per Boro ──")
df_b.groupBy("Boro").agg(
    F.countDistinct("SegmentID").alias("unique_segments"),
    F.count("*").alias("total_readings"),
    F.countDistinct(F.concat_ws("-", "Yr", "M", "D")).alias("unique_days"),
    F.countDistinct("HH").alias("unique_hours"),
).orderBy("Boro").show(truncate=False)

StatementMeta(, 17eaf14e-5bc5-48f1-a6bb-30488a9025bd, 22, Finished, Available, Finished, False)

Total Bronze rows: 9,120
Unique RequestIDs: 6
Unique SegmentIDs: 13

── Boro values (raw, exactly as stored in Bronze) ──
+---------+-----+
|Boro     |count|
+---------+-----+
|Bronx    |5376 |
|Brooklyn |2688 |
|Manhattan|672  |
|Queens   |384  |
+---------+-----+

── Unique segments per Boro ──
+---------+---------------+--------------+-----------+------------+
|Boro     |unique_segments|total_readings|unique_days|unique_hours|
+---------+---------------+--------------+-----------+------------+
|Bronx    |4              |5376          |14         |24          |
|Brooklyn |4              |2688          |7          |24          |
|Manhattan|1              |672           |7          |24          |
|Queens   |4              |384           |1          |24          |
+---------+---------------+--------------+-----------+------------+



In [21]:
# ── DIAGNOSTIC 2: Date/time coverage per Boro ────────────────────────────────

print("── Date range per Boro ──")
df_b.groupBy("Boro").agg(
    F.min(F.concat_ws("-", F.col("Yr"), F.lpad(F.col("M"),2,"0"), F.lpad(F.col("D"),2,"0"))).alias("earliest_date"),
    F.max(F.concat_ws("-", F.col("Yr"), F.lpad(F.col("M"),2,"0"), F.lpad(F.col("D"),2,"0"))).alias("latest_date"),
    F.countDistinct(F.concat_ws("-", "Yr", "M", "D")).alias("unique_days"),
    F.count("*").alias("total_readings"),
).orderBy("Boro").show(truncate=False)

print("── MM distribution (should only be 0, 15, 30, 45) ──")
df_b.groupBy("MM").count().orderBy("MM").show()

print("── HH distribution ──")
df_b.groupBy("HH").count().orderBy("HH").show(30)

StatementMeta(, 17eaf14e-5bc5-48f1-a6bb-30488a9025bd, 23, Finished, Available, Finished, False)

── Date range per Boro ──
+---------+-------------+-----------+-----------+--------------+
|Boro     |earliest_date|latest_date|unique_days|total_readings|
+---------+-------------+-----------+-----------+--------------+
|Bronx    |2026-01-20   |2026-02-02 |14         |5376          |
|Brooklyn |2026-01-03   |2026-01-09 |7          |2688          |
|Manhattan|2026-01-10   |2026-01-16 |7          |672           |
|Queens   |2026-02-03   |2026-02-03 |1          |384           |
+---------+-------------+-----------+-----------+--------------+

── MM distribution (should only be 0, 15, 30, 45) ──
+---+-----+
| MM|count|
+---+-----+
|  0| 2280|
| 15| 2280|
| 30| 2280|
| 45| 2280|
+---+-----+

── HH distribution ──
+---+-----+
| HH|count|
+---+-----+
|  0|  380|
|  1|  380|
|  2|  380|
|  3|  380|
|  4|  380|
|  5|  380|
|  6|  380|
|  7|  380|
|  8|  380|
|  9|  380|
| 10|  380|
| 11|  380|
| 12|  380|
| 13|  380|
| 14|  380|
| 15|  380|
| 16|  380|
| 17|  380|
| 18|  380|
| 19|  380|
| 20|

In [22]:
# ── DIAGNOSTIC 3: What does Silver see when it reads SegmentID → borough? ────
# This checks how Silver's borough assignment handles the Bronze Boro column.
# If Silver re-derives borough via spatial join or lookup, mismatches appear here.

print("── Sample rows per Boro — check WktGeom is populated ──")
for boro in ["Bronx", "Brooklyn", "Manhattan", "Queens", "SI", "Staten Island", "S.I."]:
    sample = df_b.filter(F.col("Boro") == boro)
    cnt = sample.count()
    if cnt > 0:
        print(f"\n  Boro='{boro}' → {cnt:,} rows — sample:")
        sample.select("RequestID", "SegmentID", "Boro", "street", "WktGeom").show(3, truncate=60)

print("\n── All distinct Boro strings (including whitespace/case variants) ──")
df_b.select(
    F.col("Boro"),
    F.length(F.col("Boro")).alias("char_len"),
    F.upper(F.trim(F.col("Boro"))).alias("normalised")
).distinct().orderBy("normalised").show(30, truncate=False)

StatementMeta(, 17eaf14e-5bc5-48f1-a6bb-30488a9025bd, 24, Finished, Available, Finished, False)

── Sample rows per Boro — check WktGeom is populated ──

  Boro='Bronx' → 5,376 rows — sample:
+---------+---------+-----+---------------+--------------------------------------------+
|RequestID|SegmentID| Boro|         street|                                     WktGeom|
+---------+---------+-----+---------------+--------------------------------------------+
|    39831|    79150|Bronx|EAST 179 STREET|POINT (1016171.2357242833 246963.5399851802)|
|    39831|    79150|Bronx|EAST 179 STREET|POINT (1016171.2357242833 246963.5399851802)|
|    39831|    79150|Bronx|EAST 179 STREET|POINT (1016171.2357242833 246963.5399851802)|
+---------+---------+-----+---------------+--------------------------------------------+
only showing top 3 rows


  Boro='Brooklyn' → 2,688 rows — sample:
+---------+---------+--------+------------------+---------------------------------------------+
|RequestID|SegmentID|    Boro|            street|                                      WktGeom|
+---------+---------+--

In [23]:
# ── DIAGNOSTIC 4: Reconstruct what Silver sees after reading Bronze ───────────
# Silver reads bronze_traffic_nyc and assigns borough — simulate that step here.

print("── Segments that Silver keeps vs drops ──")
print("   (Silver filters to only rows its borough logic can match)")

# Reconstruct Silver's timestamp from Bronze integer columns
df_sim = df_b.withColumn(
    "timestamp",
    F.to_timestamp(
        F.concat_ws("-",
            F.col("Yr").cast("string"),
            F.lpad(F.col("M").cast("string"), 2, "0"),
            F.lpad(F.col("D").cast("string"), 2, "0")
        )
        + F.lit(" ") +
        F.lpad(F.col("HH").cast("string"), 2, "0") + F.lit(":") +
        F.lpad(F.col("MM").cast("string"), 2, "0") + F.lit(":00")
    )
)

print(f"\n   Total rows Bronze provides to Silver: {df_sim.count():,}")
print(f"   Rows with NULL timestamp after reconstruction: "
      f"{df_sim.filter(F.col('timestamp').isNull()).count():,}")

print("\n── Rows per Boro after timestamp reconstruction ──")
df_sim.groupBy("Boro").agg(
    F.count("*").alias("rows"),
    F.countDistinct("SegmentID").alias("segments"),
    F.sum(F.col("timestamp").isNull().cast("int")).alias("null_timestamps"),
).orderBy("Boro").show(truncate=False)

print("\n── Vol statistics per Boro ──")
df_sim.groupBy("Boro").agg(
    F.min("Vol").alias("min_vol"),
    F.avg("Vol").cast("int").alias("avg_vol"),
    F.max("Vol").alias("max_vol"),
    F.sum(F.col("Vol").isNull().cast("int")).alias("null_vol"),
).orderBy("Boro").show(truncate=False)

StatementMeta(, 17eaf14e-5bc5-48f1-a6bb-30488a9025bd, 25, Finished, Available, Finished, False)

── Segments that Silver keeps vs drops ──
   (Silver filters to only rows its borough logic can match)

   Total rows Bronze provides to Silver: 9,120
   Rows with NULL timestamp after reconstruction: 9,120

── Rows per Boro after timestamp reconstruction ──
+---------+----+--------+---------------+
|Boro     |rows|segments|null_timestamps|
+---------+----+--------+---------------+
|Bronx    |5376|4       |5376           |
|Brooklyn |2688|4       |2688           |
|Manhattan|672 |1       |672            |
|Queens   |384 |4       |384            |
+---------+----+--------+---------------+


── Vol statistics per Boro ──
+---------+-------+-------+-------+--------+
|Boro     |min_vol|avg_vol|max_vol|null_vol|
+---------+-------+-------+-------+--------+
|Bronx    |0      |34     |277    |0       |
|Brooklyn |0      |45     |184    |0       |
|Manhattan|0      |23     |78     |0       |
|Queens   |3      |64     |197    |0       |
+---------+-------+-------+-------+--------+

